In [33]:
import re
import zipfile
import pandas as pd
from pathlib import Path
from itertools import combinations


In [19]:
def kaggle_data_handler(kaggle_url, csv_index=0):
    match = re.search(r"kaggle\.com/datasets/([^/]+)/([^/?#]+)", kaggle_url)
    if not match:
        raise ValueError(f"Could not parse Kaggle dataset URL: {kaggle_url}")

    owner, slug = match.group(1), match.group(2)

    try:
        download_dir = Path(__file__).parent / "data"
    except NameError:
        download_dir = Path.cwd() / "data"

    existing_csvs = sorted(download_dir.glob("*.csv")) if download_dir.exists() else []

    if not existing_csvs:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        download_dir.mkdir(parents=True, exist_ok=True)
        api.dataset_download_files(f"{owner}/{slug}", path=str(download_dir), unzip=False, quiet=False)

        for z in download_dir.glob("*.zip"):
            with zipfile.ZipFile(z, "r") as zf:
                zf.extractall(download_dir)
            z.unlink()

        existing_csvs = sorted(download_dir.glob("*.csv"))

    return pd.read_csv(existing_csvs[csv_index])

In [20]:
kaggle_link = 'https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows'

In [21]:
df = kaggle_data_handler(kaggle_link)

Dataset URL: https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows


100%|██████████| 175k/175k [00:00<00:00, 658kB/s]

In [23]:
df.describe()

,IMDB_Rating,Meta_score,No_of_Votes
count,1000.000000,843.000000,1.000000e+03
mean,7.949300,77.971530,2.736929e+05
std,0.275491,12.376099,3.273727e+05
min,7.600000,28.000000,2.508800e+04
25%,7.700000,70.000000,5.552625e+04
50%,7.900000,79.000000,1.385485e+05
75%,8.100000,87.000000,3.741612e+05
max,9.300000,100.000000,2.343110e+06


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Poster_Link    1000 non-null   str    
 1   Series_Title   1000 non-null   str    
 2   Released_Year  1000 non-null   str    
 3   Certificate    899 non-null    str    
 4   Runtime        1000 non-null   str    
 5   Genre          1000 non-null   str    
 6   IMDB_Rating    1000 non-null   float64
 7   Overview       1000 non-null   str    
 8   Meta_score     843 non-null    float64
 9   Director       1000 non-null   str    
 10  Star1          1000 non-null   str    
 11  Star2          1000 non-null   str    
 12  Star3          1000 non-null   str    
 13  Star4          1000 non-null   str    
 14  No_of_Votes    1000 non-null   int64  
 15  Gross          831 non-null    str    
dtypes: float64(2), int64(1), str(13)
memory usage: 125.1 KB


In [25]:
df.columns

Index(['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate',
       'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director',
       'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross'],
      dtype='str')

In [28]:
baskets = df[["Star1","Star2","Star3","Star4"]].values.tolist()
baskets = [set(row) for row in baskets]

In [29]:
baskets

[{'Bob Gunton', 'Morgan Freeman', 'Tim Robbins', 'William Sadler'},
 {'Al Pacino', 'Diane Keaton', 'James Caan', 'Marlon Brando'},
 {'Aaron Eckhart', 'Christian Bale', 'Heath Ledger', 'Michael Caine'},
 {'Al Pacino', 'Diane Keaton', 'Robert De Niro', 'Robert Duvall'},
 {'Henry Fonda', 'John Fiedler', 'Lee J. Cobb', 'Martin Balsam'},
 {'Elijah Wood', 'Ian McKellen', 'Orlando Bloom', 'Viggo Mortensen'},
 {'Bruce Willis', 'John Travolta', 'Samuel L. Jackson', 'Uma Thurman'},
 {'Ben Kingsley', 'Caroline Goodall', 'Liam Neeson', 'Ralph Fiennes'},
 {'Elliot Page', 'Joseph Gordon-Levitt', 'Ken Watanabe', 'Leonardo DiCaprio'},
 {'Brad Pitt', 'Edward Norton', 'Meat Loaf', 'Zach Grenier'},
 {'Elijah Wood', 'Ian McKellen', 'Orlando Bloom', 'Sean Bean'},
 {'Gary Sinise', 'Robin Wright', 'Sally Field', 'Tom Hanks'},
 {'Aldo Giuffrè', 'Clint Eastwood', 'Eli Wallach', 'Lee Van Cleef'},
 {'Elijah Wood', 'Ian McKellen', 'Orlando Bloom', 'Viggo Mortensen'},
 {'Carrie-Anne Moss', 'Keanu Reeves', 'Laurenc

In [39]:
singleton_count = {}

for basket in baskets:
    for actor in basket:
        singleton_count[actor] = singleton_count.get(actor, 0) + 1

THRESHOLD = 2

frequent_singletons = {
    actor for actor, count in singleton_count.items()
    if count >= THRESHOLD
}

In [40]:
print("Top frequent singletons:")
for actor, count in sorted(singleton_count.items(), key=lambda x: -x[1])[:10]:
    print(f"  {actor:30s}  support={count}")

Top frequent singletons:
  Robert De Niro                  support=17
  Tom Hanks                       support=14
  Al Pacino                       support=13
  Brad Pitt                       support=12
  Clint Eastwood                  support=12
  Christian Bale                  support=11
  Leonardo DiCaprio               support=11
  Matt Damon                      support=11
  James Stewart                   support=10
  Michael Caine                   support=9


In [41]:
pair_count = {}

for basket in baskets:
    actors = list(basket)
    for a, b in combinations(sorted(actors), 2):
        key = (a, b)
        pair_count[key] = pair_count.get(key, 0) + 1

frequent_pairs = {
    pair: count for pair, count in pair_count.items()
    if count >= THRESHOLD
}


In [42]:
print("\nFrequent pairs with confidence:")
for (a, b), count in sorted(frequent_pairs.items(), key=lambda x: -x[1]):
    conf_ab = count / singleton_count[a]
    conf_ba = count / singleton_count[b]
    print(f"  {a} & {b}")
    print(f"    support={count}  conf({a}→{b})={conf_ab:.2f}  conf({b}→{a})={conf_ba:.2f}")



Frequent pairs with confidence:
  Daniel Radcliffe & Rupert Grint
    support=6  conf(Daniel Radcliffe→Rupert Grint)=1.00  conf(Rupert Grint→Daniel Radcliffe)=1.00
  Daniel Radcliffe & Emma Watson
    support=5  conf(Daniel Radcliffe→Emma Watson)=0.83  conf(Emma Watson→Daniel Radcliffe)=0.71
  Emma Watson & Rupert Grint
    support=5  conf(Emma Watson→Rupert Grint)=0.71  conf(Rupert Grint→Emma Watson)=0.83
  Joe Pesci & Robert De Niro
    support=4  conf(Joe Pesci→Robert De Niro)=0.67  conf(Robert De Niro→Joe Pesci)=0.24
  Tim Allen & Tom Hanks
    support=4  conf(Tim Allen→Tom Hanks)=1.00  conf(Tom Hanks→Tim Allen)=0.29
  Al Pacino & Diane Keaton
    support=3  conf(Al Pacino→Diane Keaton)=0.23  conf(Diane Keaton→Al Pacino)=0.50
  Christian Bale & Michael Caine
    support=3  conf(Christian Bale→Michael Caine)=0.27  conf(Michael Caine→Christian Bale)=0.33
  Al Pacino & Robert De Niro
    support=3  conf(Al Pacino→Robert De Niro)=0.23  conf(Robert De Niro→Al Pacino)=0.18
  Elijah Wood

In [43]:
print("\nProof — every frequent pair must have two frequent singletons:")
for (a, b), count in frequent_pairs.items():
    a_ok = a in frequent_singletons
    b_ok = b in frequent_singletons
    print(f"  ({a}, {b})  →  {a}: {'✓' if a_ok else '✗'}  {b}: {'✓' if b_ok else '✗'}")


Proof — every frequent pair must have two frequent singletons:
  (Al Pacino, Diane Keaton)  →  Al Pacino: ✓  Diane Keaton: ✓
  (Christian Bale, Michael Caine)  →  Christian Bale: ✓  Michael Caine: ✓
  (Al Pacino, Robert De Niro)  →  Al Pacino: ✓  Robert De Niro: ✓
  (Elijah Wood, Ian McKellen)  →  Elijah Wood: ✓  Ian McKellen: ✓
  (Elijah Wood, Orlando Bloom)  →  Elijah Wood: ✓  Orlando Bloom: ✓
  (Elijah Wood, Viggo Mortensen)  →  Elijah Wood: ✓  Viggo Mortensen: ✓
  (Ian McKellen, Orlando Bloom)  →  Ian McKellen: ✓  Orlando Bloom: ✓
  (Ian McKellen, Viggo Mortensen)  →  Ian McKellen: ✓  Viggo Mortensen: ✓
  (Orlando Bloom, Viggo Mortensen)  →  Orlando Bloom: ✓  Viggo Mortensen: ✓
  (Bruce Willis, Samuel L. Jackson)  →  Bruce Willis: ✓  Samuel L. Jackson: ✓
  (Gary Sinise, Tom Hanks)  →  Gary Sinise: ✓  Tom Hanks: ✓
  (Clint Eastwood, Lee Van Cleef)  →  Clint Eastwood: ✓  Lee Van Cleef: ✓
  (Joe Pesci, Robert De Niro)  →  Joe Pesci: ✓  Robert De Niro: ✓
  (Billy Dee Williams, Carrie 